# Chapter 4 — Seller Analysis
Queries `mart_seller_analysis` from BigQuery and exports Plotly chart JSON for the webpage.

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery
from google.oauth2 import service_account

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(project_root, '.env'))

project_id  = os.getenv('GCP_PROJECT_ID')
creds_path  = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials = service_account.Credentials.from_service_account_file(creds_path)
client      = bigquery.Client(credentials=credentials, project=project_id)

OUT = os.path.join(project_root, 'outputs')
os.makedirs(OUT, exist_ok=True)
print('Connected to BigQuery ✓')

In [ ]:
df = client.query(f"""
    SELECT * FROM `{project_id}.olist_raw.mart_seller_analysis`
    ORDER BY total_revenue DESC
""").to_dataframe()
df.head()

In [ ]:
# Chart 1 — Top 20 Sellers by Revenue
top_sellers = df.head(20).copy()
top_sellers['seller_label'] = top_sellers['seller_id'].str[:8] + '...'

fig1 = px.bar(
    top_sellers, x='total_revenue', y='seller_label', orientation='h',
    title='Top 20 Sellers by Revenue (BRL)',
    labels={'total_revenue': 'Revenue (BRL)', 'seller_label': 'Seller'},
    color='avg_review_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5]
)
fig1.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'})
fig1.show()
with open(os.path.join(OUT, 'seller_top20_revenue.json'), 'w') as f:
    f.write(fig1.to_json())
print('Exported seller_top20_revenue.json')

In [ ]:
# Chart 2 — Seller Count and Revenue by State
by_state = df.groupby('seller_state').agg(
    seller_count=('seller_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_review=('avg_review_score', 'mean'),
    avg_on_time=('on_time_pct', 'mean')
).reset_index().sort_values('total_revenue', ascending=False)

fig2 = px.bar(
    by_state, x='seller_state', y='total_revenue',
    title='Total Seller Revenue by State (BRL)',
    labels={'seller_state': 'State', 'total_revenue': 'Revenue (BRL)'},
    color='seller_count',
    color_continuous_scale='Blues'
)
fig2.update_layout(template='plotly_white')
fig2.show()
with open(os.path.join(OUT, 'seller_revenue_by_state.json'), 'w') as f:
    f.write(fig2.to_json())
print('Exported seller_revenue_by_state.json')

In [ ]:
# Chart 3 — Review Score vs On-Time % (top 200 sellers by revenue)
top200 = df.head(200)
fig3 = px.scatter(
    top200, x='on_time_pct', y='avg_review_score',
    size='total_revenue',
    color='seller_state',
    title='Review Score vs On-Time % (Top 200 Sellers)',
    labels={'on_time_pct': 'On-Time %', 'avg_review_score': 'Avg Review Score', 'total_revenue': 'Revenue'}
)
fig3.update_layout(template='plotly_white')
fig3.show()
with open(os.path.join(OUT, 'seller_score_vs_ontime.json'), 'w') as f:
    f.write(fig3.to_json())
print('Exported seller_score_vs_ontime.json')

In [ ]:
# Chart 4 — Seller State Distribution (pie)
fig4 = px.pie(
    by_state, names='seller_state', values='seller_count',
    title='Seller Distribution by State',
    hole=0.4
)
fig4.update_layout(template='plotly_white')
fig4.show()
with open(os.path.join(OUT, 'seller_state_distribution.json'), 'w') as f:
    f.write(fig4.to_json())
print('Exported seller_state_distribution.json')